<a href="https://colab.research.google.com/github/jeeen0/cv-2026-lab/blob/main/ex08_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EX.08-1

# 8-1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# load iris
dataset = load_iris()
feature = dataset.data
label = dataset.target
print(dataset.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [ ]:
# define net.
# https://pytorch.org/docs/stable/nn.html#
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        n = 8 # 4

        # fully-connected layers
        self.inputLayer   = nn.Linear(3, n) # 입력 노드 수 변경
        self.hiddenLayer1 = nn.Linear(n, n) # Hidden layer 1
        self.hiddenLayer2 = nn.Linear(n, n) # Hidden layer 2
        self.outputLayer  = nn.Linear(n, 3) # 출력 노드 수 변경

        # batch normalization
        self.bn = nn.BatchNorm1d(n)

        # activation function
        self.act = nn.ReLU()
        #self.act = nn.Tanh() # 활성화 함수 변경?

    def forward(self, x): # 네트워크 구성
        x = self.act(self.bn(self.inputLayer(x)))
        x = self.act(self.bn(self.hiddenLayer1(x)))
        x = self.act(self.bn(self.hiddenLayer2(x)))
        x = self.outputLayer(x)
        return x

In [ ]:
net = Net()
print(net)

Net(
  (inputLayer): Linear(in_features=3, out_features=8, bias=True)
  (hiddenLayer1): Linear(in_features=8, out_features=8, bias=True)
  (hiddenLayer2): Linear(in_features=8, out_features=8, bias=True)
  (outputLayer): Linear(in_features=8, out_features=3, bias=True)
  (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act): ReLU()
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)
#criterion = nn.NLLLoss() # 비용 함수 변경 (e.g. log likelihood)?
print(criterion)
print(optimizer)

CrossEntropyLoss()
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [ ]:
# prep. train/test data
X_train, X_test, y_train, y_test = train_test_split(feature, label, test_size=0.5)
print('# of train samples:', len(X_train), ',', '# of test samples:', len(X_test))

# of train samples: 75 , # of test samples: 75


In [ ]:
# exclud. the 2nd feature??
X_train_new = np.column_stack((X_train[:,0], X_train[:,2:]))
X_test_new = np.column_stack((X_test[:,0], X_test[:,2:]))
X_train_new = torch.from_numpy(X_train_new).float()
X_test_new = torch.from_numpy(X_test_new).float()
#X_train = torch.from_numpy(X_train).float()
#X_test = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

In [ ]:
#train_set = TensorDataset(X_train, y_train)
trainSet = TensorDataset(X_train_new, y_train)
trainLoader = DataLoader(trainSet, batch_size=25, shuffle=True)

In [ ]:
for epoch in range(100):  # loop over the dataset multiple times

    for i, data in enumerate(trainLoader):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # evaluation
        _, preds = torch.max(outputs, dim=1)
        print('[%d, %d]:' % (epoch+1, i+1), 'loss:', loss.item(), ',', '# of true positives', sum(preds == labels).item())

[1, 1]: loss: 1.2327438592910767 , # of true positives 9
[1, 2]: loss: 1.1859697103500366 , # of true positives 11
[1, 3]: loss: 1.1140716075897217 , # of true positives 17
[2, 1]: loss: 1.1071048974990845 , # of true positives 17
[2, 2]: loss: 1.061859130859375 , # of true positives 16
[2, 3]: loss: 1.2034826278686523 , # of true positives 10
[3, 1]: loss: 1.1423531770706177 , # of true positives 13
[3, 2]: loss: 0.9534781575202942 , # of true positives 18
[3, 3]: loss: 1.1139943599700928 , # of true positives 13
[4, 1]: loss: 1.0184364318847656 , # of true positives 17
[4, 2]: loss: 1.136420726776123 , # of true positives 11
[4, 3]: loss: 1.0445088148117065 , # of true positives 17
[5, 1]: loss: 1.0288540124893188 , # of true positives 15
[5, 2]: loss: 1.0604369640350342 , # of true positives 16
[5, 3]: loss: 1.0147144794464111 , # of true positives 18
[6, 1]: loss: 1.0135607719421387 , # of true positives 17
[6, 2]: loss: 0.9534909129142761 , # of true positives 18
[6, 3]: loss: 1.0

In [ ]:
# Test
#outputs = net(X_test)
outputs = net(X_test_new)
_, preds = torch.max(outputs, dim=1)
eval = sum(preds == y_test).item() / len(y_test)
print('eval.: ', eval)

eval.:  0.9466666666666667
